<a href="https://colab.research.google.com/github/tomarjyoti16/assignment-05-bitsom_ba_2511563_part-2-cnn-computer-vision/blob/main/Computer_Vision_Problem_Formulation_and_CNN_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Problem Identification

The dataset represents an Image classification problem.
* Single Label Per Image: The dataset maps each complete image file directly to exactly one discrete categorical label (normal, scratch, dent, or stain) via a flat subfolder layout or a global metadata file (labels.csv).
* Lack of Spatial Annotations: The provided file structure does not contain spatial coordinates, bounding box coordinates (needed for object detection), or pixel-level binary masks (needed for semantic or instance segmentation).
* Global Scene Prediction: The goal of the convolutional neural network (CNN) is to evaluate the visual properties of the entire product surface at once and output a single holistic prediction category rather than isolating or locating individual flaw zones.

Task 2: Dataset Exploration

In [ ]:
import os
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# Define configuration
IMAGE_DIR = "images/"
CLASSES = ["normal", "scratch", "dent", "stain"]

# 1. Gather Dataset Metadata
data_records = []
for class_name in CLASSES:
    class_path = os.path.join(IMAGE_DIR, class_name)
    if os.path.exists(class_path):
        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in files:
            full_path = os.path.join(class_path, f)
            # Read image metadata efficiently without loading entire content to memory
            img = cv2.imread(full_path)
            if img is not None:
                h, w, c = img.shape
                data_records.append({
                    "filename": f,
                    "class": class_name,
                    "height": h,
                    "width": w,
                    "channels": c
                })

df = pd.DataFrame(data_records)

# 2. Analysis & Report Generation
print("=== DATASET EXPLORATION REPORT ===")
print(f"Number of Classes: {df['class'].nunique()}")

print("\n--- Number of Images Per Class ---")
counts = df["class"].value_counts()
print(counts.to_string())

print("\n--- Image Dimensions ---")
unique_shapes = df.groupby(["height", "width", "channels"]).size().reset_index(name="count")
print(unique_shapes.to_string(index=False))

print("\n--- Dataset Imbalance Check ---")
min_class = counts.min()
max_class = counts.max()
imbalance_ratio = max_class / min_class
print(f"Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}x")
if imbalance_ratio > 1.5:
    print("Warning: Dataset is significantly imbalanced. Consider class-weighting or augmentation.")
else:
    print("Dataset is relatively well-balanced.")

# 3. Visualize Sample Images From Each Class
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
for i, class_name in enumerate(CLASSES):
    class_df = df[df["class"] == class_name]
    if not class_df.empty:
        sample_row = class_df.iloc[0]
        sample_path = os.path.join(IMAGE_DIR, class_name, sample_row["filename"])
        img = cv2.imread(sample_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[i].imshow(img_rgb)
        axes[i].set_title(f"{class_name.capitalize()}\n({sample_row['height']}x{sample_row['width']})")
        axes[i].axis("off")
plt.tight_layout()
plt.show()


Task 3: Image Preprocessing

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Define preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((128, 128)),       # Resize to fixed shape
    transforms.ToTensor(),               # Converts to PyTorch Tensor
    transforms.Normalize(                # Normalize pixel values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load dataset from folder structure
dataset = ImageFolder(root='images/', transform=transform)

# Split into train and validation sets (80/20 split)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Classes found: {dataset.classes}")

Task 4: CNN Model Creation

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DefectClassifier(nn.Module):
    def __init__(self):
        super(DefectClassifier, self).__init__()
        # Input: 3 x 128 x 128
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) # Reduces dimensions by half

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # After 3 pooling layers, 128x128 becomes 16x16
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 4) # 4 output classes
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(-1, 64 * 16 * 16) # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = DefectClassifier()

Task 5: Model Training and Evaluation

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

Task 6: CNN Concept Explanation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    # Unnormalize image for visualization
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get a batch of validation images
dataiter = iter(val_loader)
images, labels = next(dataiter)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(images.to(device))
    _, predicted = torch.max(outputs, 1)

# Display first sample
imshow(images[0])
print(f"Ground Truth: {dataset.classes[labels[0]]}")
print(f"Prediction: {dataset.classes[predicted[0]]}")

Task 7: Business Use Case Mapping

Manufacturing Quality Inspection Use Case
* Automated visual sorting replaces manual inspection on high-speed factory production lines.
* Production Deployment Workflow
-  In-Line Image Capture: High-speed industrial cameras capture uniform top-down photos of items moving down a conveyor belt.
- Microsecond Edge Inference: A local computing device passes each image through the trained CNN model within milliseconds.
- Automated Product Sorting: The model output instantly triggers mechanical hardware switches on the production floor:
- normal products continue down the main line toward packaging.
- scratch and dent products trigger pneumatic arms to divert items into a hardware rework bin.
- stain products route to an industrial cleaning station.
* Concrete Factory BenefitsRoot Cause Diagnostics:
- A sudden spike in stain predictions alerts plant engineers to clean chemical bath filters, while a surge in scratch flags friction wear on robotic transport rails.
- Escaping Customer Defects: The continuous system eliminates human operator fatigue, preventing structural micro-defects from escaping to shipping and avoiding costly warranty claims.

Sample Predictions

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def visualize_predictions(model, val_loader, class_names, num_images=4):
    """
    Takes a trained model and validation loader, runs inference on a batch,
    and plots sample images with True vs. Predicted labels.
    """
    # Set model to evaluation mode
    model.eval()

    # Get a single batch of images and labels from the validation loader
    images, labels = next(iter(val_loader))

    # Move inputs to the active device (GPU or CPU)
    device = next(model.parameters()).device
    images_dev = images.to(device)

    # Disable gradient tracking for speed and memory efficiency
    with torch.no_grad():
        outputs = model(images_dev)
        _, preds = torch.max(outputs, 1)

    # Move back to CPU for numpy operations and visualization
    images = images.cpu()
    labels = labels.cpu().numpy()
    preds = preds.cpu().numpy()

    # Plot configuration
    fig, axes = plt.subplots(1, num_images, figsize=(16, 4))

    # Ensure axes is always iterable even if num_images is 1
    if num_images == 1:
        axes = [axes]

    for i in range(num_images):
        # Unnormalize the image tensor for proper visual display
        img = images[i].numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1) # Force pixel ranges to [0, 1]

        # Determine label color: green if correct, red if incorrect
        is_correct = preds[i] == labels[i]
        title_color = 'green' if is_correct else 'red'

        # Display image
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(
            f"True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}",
            color=title_color,
            fontsize=12,
            weight='bold'
        )

    plt.tight_layout()
    plt.show()

# --- Execution Example ---
# Assuming 'model', 'val_loader', and 'dataset' are defined from your previous tasks:
# class_names = dataset.classes # ['normal', 'scratch', 'dent', 'stain']
# visualize_predictions(model, val_loader, class_names, num_images=4)